# 🚗 Road Accident Severity Prediction — CNN
### RTX 5080 Optimized | EfficientNetV2-S | Transfer Learning | 3-Class Severity

---

**Dataset folder expected (same directory as this notebook):**
```
dataset3/
  train/
    Accident/        ← accident images
    Non Accident/    ← non-accident images
  test/
    Accident/
    Non Accident/
```
The notebook auto-scans `dataset3/` so slight variations in structure are handled automatically.


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu128')
pip('timm')
pip('albumentations')
pip('opencv-python')
pip('matplotlib', 'seaborn')
pip('scikit-learn')
pip('tqdm')
pip('grad-cam')
pip('pandas')
pip('pillow')

print('✅ All packages installed!')

## Cell 2 — Imports & GPU Setup

In [ ]:
import os, shutil, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score
)
from sklearn.preprocessing import label_binarize

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = True   # RTX 5080: perf boost

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram     = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    print(f'🚀 GPU   : {gpu_name}')
    print(f'   VRAM  : {vram:.1f} GB')
    print(f'   CUDA  : {torch.version.cuda}')
    print(f'   TF32  : enabled')
else:
    print('⚠️  No GPU — running on CPU (will be slow)')

print(f'\n📌 Device : {DEVICE}')

## Cell 3 — Config & Paths

In [ ]:
# ── Paths  ────────────────────────────────────────────────────────────────────
#   dataset3/ is in the same folder as this notebook
RAW_DIR   = Path('dataset3')       # ← change this if your folder has a different name
PROC_DIR  = Path('data/processed') # restructured 3-class dataset (auto-created)
CKPT_DIR  = Path('checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['No_Accident', 'Minor_Accident', 'Severe_Accident']
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_COLORS = ['#2ecc71', '#f39c12', '#e74c3c']

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'tf_efficientnetv2_s'
IMG_SIZE    = 224
PRETRAINED  = True

# ── Training  ─────────────────────────────────────────────────────────────────
BATCH_SIZE     = 64      # fits RTX 5080 (16 GB) at 224px
NUM_EPOCHS     = 30
LR_HEAD        = 3e-4
LR_BACKBONE    = 5e-5
WEIGHT_DECAY   = 1e-4
PATIENCE       = 7
UNFREEZE_EPOCH = 5
USE_AMP        = True
NUM_WORKERS    = 0       # 0 = safe default on Windows; raise to 4-8 if no errors
LABEL_SMOOTH   = 0.1

print('Config loaded ✅')
print(f'  RAW_DIR    : {RAW_DIR.resolve()}')
print(f'  PROC_DIR   : {PROC_DIR.resolve()}')
print(f'  Model      : {MODEL_NAME}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  AMP        : {USE_AMP}')
print(f'  Workers    : {NUM_WORKERS}')

## Cell 4 — Inspect dataset3 Structure (run this first!)

In [ ]:
def print_tree(root: Path, max_depth: int = 4, _depth: int = 0, _prefix: str = ''):
    if _depth > max_depth:
        return
    try:
        entries = sorted(root.iterdir())
    except PermissionError:
        return
    for i, entry in enumerate(entries):
        connector = '└── ' if i == len(entries) - 1 else '├── '
        suffix = ''
        if entry.is_dir():
            imgs = []
            for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
                imgs += list(entry.glob(ext))
            if imgs:
                suffix = f'  [{len(imgs)} images]'
        print(_prefix + connector + entry.name + suffix)
        if entry.is_dir():
            ext_prefix = '    ' if i == len(entries) - 1 else '│   '
            print_tree(entry, max_depth, _depth + 1, _prefix + ext_prefix)

if RAW_DIR.exists():
    print(f'{RAW_DIR}/')
    print_tree(RAW_DIR, max_depth=4)
else:
    print(f'❌  {RAW_DIR.resolve()} does NOT exist!')
    print(f'   CWD: {Path.cwd()}')
    print()
    print('Searching for "Accident" folders anywhere nearby...')
    for p in sorted(Path('.').rglob('Accident')):
        print(f'  Found: {p}')

## Cell 5 — Build 3-Class Dataset from dataset3

In [ ]:
import stat

def force_remove_readonly(func, path, _):
    """Error handler for rmtree — clears read-only flag then retries."""
    os.chmod(path, stat.S_IWRITE)
    func(path)

def safe_rmtree(path: Path):
    """Windows-safe recursive delete."""
    if path.exists():
        shutil.rmtree(path, onerror=force_remove_readonly)


def laplacian_variance(img_path: Path) -> float:
    """Blur metric — lower variance = blurrier = more chaotic crash frame."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    return cv2.Laplacian(img, cv2.CV_64F).var() if img is not None else 0.0


def collect_images(search_root: Path, class_folder_name: str) -> list:
    """
    Recursively find all images inside any folder whose name matches
    class_folder_name (case-insensitive). Handles both:
      dataset3/train/Accident/  and  dataset3/Accident/  etc.
    """
    imgs = []
    for d in search_root.rglob('*'):
        if d.is_dir() and d.name.lower() == class_folder_name.lower():
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                imgs.extend(d.glob(ext))
    return imgs


def split_copy(imgs: list, cls_name: str, out_dir: Path,
               ratios=(0.75, 0.15, 0.10)):
    random.shuffle(imgs)
    n  = len(imgs)
    tr = int(n * ratios[0])
    vl = int(n * ratios[1])
    buckets = {
        'train': imgs[:tr],
        'val'  : imgs[tr:tr+vl],
        'test' : imgs[tr+vl:]
    }
    for split, files in buckets.items():
        dest = out_dir / split / cls_name
        dest.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(files):
            shutil.copy(f, dest / f'{cls_name}_{i:05d}{f.suffix}')
    print(f'  {cls_name:<22}  train={len(buckets["train"]):>4}  '
          f'val={len(buckets["val"]):>4}  test={len(buckets["test"]):>4}')
    return buckets


def has_complete_dataset(out_dir: Path) -> bool:
    extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    return all(
        any(path.suffix.lower() in extensions for path in (out_dir / split / cls).glob('*'))
        for split in ('train', 'val', 'test')
        for cls in CLASS_NAMES
    )


def build_dataset(raw_dir: Path, out_dir: Path, force: bool = False):
    if out_dir.exists() and not force and has_complete_dataset(out_dir):
        total = sum(1 for path in out_dir.rglob('*')
                    if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
        print(f'? Complete processed dataset already exists ({total:,} images). '
              f'Pass force=True to rebuild.')
        return

    safe_rmtree(out_dir)       # Rebuild incomplete processed datasets.

    if not raw_dir.exists():
        raise FileNotFoundError(
            f'RAW_DIR not found: {raw_dir.resolve()}\n'
            f'Make sure dataset3/ is in the same folder as this notebook.'
        )

    # ── Collect raw images ────────────────────────────────────────────────────
    print(f'Scanning {raw_dir} ...')
    accident_imgs     = collect_images(raw_dir, 'Accident')
    non_accident_imgs = collect_images(raw_dir, 'Non Accident')

    # De-duplicate (some datasets have overlapping train/test folders)
    accident_imgs     = list({str(p): p for p in accident_imgs}.values())
    non_accident_imgs = list({str(p): p for p in non_accident_imgs}.values())

    print(f'  Accident images     : {len(accident_imgs):,}')
    print(f'  Non-Accident images : {len(non_accident_imgs):,}')

    if len(accident_imgs) == 0:
        raise ValueError(
            'No accident images found! Run Cell 4 to inspect your folder structure.'
        )

    # ── No_Accident ───────────────────────────────────────────────────────────
    print('\nCopying files...')
    split_copy(non_accident_imgs, 'No_Accident', out_dir)

    # ── Minor / Severe via Laplacian blur heuristic ───────────────────────────
    print(f'Computing blur scores for {len(accident_imgs)} accident frames...')
    scored = [
        (img, laplacian_variance(img))
        for img in tqdm(accident_imgs, desc='Blur scoring')
    ]
    scored.sort(key=lambda x: x[1])   # low var → blurrier → Severe
    half         = len(scored) // 2
    severe_imgs  = [x[0] for x in scored[:half]]
    minor_imgs   = [x[0] for x in scored[half:]]

    split_copy(severe_imgs, 'Severe_Accident', out_dir)
    split_copy(minor_imgs,  'Minor_Accident',  out_dir)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f'\n✅ Dataset built at  {out_dir.resolve()}')
    grand_total = 0
    for split in ('train', 'val', 'test'):
        print(f'  {split.upper()}')
        for cls in CLASS_NAMES:
            cls_dir = out_dir / split / cls
            n = len(list(cls_dir.glob('*'))) if cls_dir.exists() else 0
            grand_total += n
            bar = '█' * (n // 15)
            print(f'    {cls:<22} {n:>5}  {bar}')
    print(f'\n  Grand total : {grand_total:,} images')


build_dataset(RAW_DIR, PROC_DIR, force=False)

## Cell 6 — Augmentation Pipelines

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=15, p=0.5),
    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=3),
        A.MedianBlur(blur_limit=3),
    ], p=0.3),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20),
        A.CLAHE(clip_limit=4.0),
    ], p=0.5),
    A.RandomRain(p=0.1),
    A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.1),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('Augmentation pipelines ready ✅')

## Cell 7 — Dataset Class & DataLoaders

In [ ]:
class AccidentDataset(Dataset):
    def __init__(self, root: Path, split: str, transform=None):
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.samples      = []

        for cls in CLASS_NAMES:
            cls_dir = root / split / cls
            if not cls_dir.exists():
                print(f'  ⚠️  {cls_dir} not found — skipping')
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                for img_path in cls_dir.glob(ext):
                    self.samples.append((str(img_path), self.class_to_idx[cls]))

        print(f'  [{split:5s}]  {len(self.samples):>6,} images  '
              f'| classes: {[sum(1 for _,l in self.samples if l==i) for i in range(NUM_CLASSES)]}')

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            # Fallback: return black image if file is corrupt
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


print('Building dataset splits...')
train_ds = AccidentDataset(PROC_DIR, 'train', train_transform)
val_ds   = AccidentDataset(PROC_DIR, 'val',   val_transform)
test_ds  = AccidentDataset(PROC_DIR, 'test',  val_transform)

# ── Guard ─────────────────────────────────────────────────────────────────────
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(
            f'?  {name} split has 0 images!\n'
            f'    Re-run Cell 4 to inspect the folder structure, '
            f'then Cell 5 with force=True.'
        )
    split_counts = np.bincount([label for _, label in ds.samples], minlength=NUM_CLASSES)
    missing_classes = [CLASS_NAMES[i] for i, count in enumerate(split_counts) if count == 0]
    if missing_classes:
        raise RuntimeError(
            f'?  {name} split is missing classes: {missing_classes}. '
            'Rebuild the processed dataset before training.'
        )

# ?? Class weights (handle class imbalance) ????????????????????????????????????
label_counts = np.zeros(NUM_CLASSES)
for _, lbl in train_ds.samples:
    label_counts[lbl] += 1
if np.any(label_counts == 0):
    missing_classes = [CLASS_NAMES[i] for i, count in enumerate(label_counts) if count == 0]
    raise RuntimeError(f'Cannot calculate class weights; no training images for: {missing_classes}')
class_weights = torch.tensor(
    len(train_ds.samples) / (NUM_CLASSES * label_counts),
    dtype=torch.float32
).to(DEVICE)
print(f'\nClass weights : {dict(zip(CLASS_NAMES, class_weights.cpu().numpy().round(3)))}')

# ── DataLoaders ───────────────────────────────────────────────────────────────
#   NUM_WORKERS=0 is safest on Windows (avoids multiprocessing issues)
#   Increase to 4 or 8 once everything runs without errors
loader_kwargs = dict(
    batch_size        = BATCH_SIZE,
    num_workers       = NUM_WORKERS,
    pin_memory        = DEVICE.type == 'cuda',
    persistent_workers= NUM_WORKERS > 0,
)
train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')
print(f'\n✅ DataLoaders ready!')

## Cell 8 — Visualise Sample Batch

In [ ]:
def denorm(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))

# Collect up to 8 examples per class
per_class = {0: [], 1: [], 2: []}
for img, lbl in zip(images, labels):
    l = lbl.item()
    if len(per_class[l]) < 8:
        per_class[l].append(img)

cols = max(len(v) for v in per_class.values())
rows = NUM_CLASSES
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
fig.suptitle('Sample Training Batch  (after augmentation)', fontsize=13, fontweight='bold')

for row, cls_idx in enumerate(range(NUM_CLASSES)):
    for col in range(cols):
        ax = axes[row][col]
        if col < len(per_class[cls_idx]):
            npimg = denorm(per_class[cls_idx][col]).permute(1,2,0).numpy()
            ax.imshow(npimg)
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[cls_idx], color=CLASS_COLORS[cls_idx],
                              fontsize=8, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('sample_batch.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → sample_batch.png')

## Cell 9 — Model (EfficientNetV2-S + Custom Head)

In [ ]:
class AccidentCNN(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True, dropout=0.35):
        super().__init__()
        # Backbone — pretrained, head removed
        self.backbone  = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        feat_dim       = self.backbone.num_features

        # Custom classifier head
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        # Weight init
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print('🔒 Backbone frozen  — training head only')

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print('🔓 Backbone unfrozen — fine-tuning entire network')

    def forward(self, x):
        feats = self.backbone(x)   # (B, feat_dim)  — timm num_classes=0 global-pools
        if feats.dim() == 4:       # safety: some configs return (B,C,H,W)
            feats = feats.mean(dim=(2, 3))
        return self.classifier(feats)


model = AccidentCNN(MODEL_NAME, NUM_CLASSES, PRETRAINED).to(DEVICE)

# torch.compile — massive throughput gain on RTX 5080 (requires PyTorch 2.x)


total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params     : {total/1e6:.2f} M')
print(f'Trainable params : {trainable/1e6:.2f} M')

## Cell 10 — Loss, Optimiser, Scheduler

In [ ]:
def get_base(m):
    """Unwrap torch.compile wrapper to access backbone/classifier attributes."""
    return m._orig_mod if hasattr(m, '_orig_mod') else m

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
scaler    = GradScaler(enabled=USE_AMP)

# Phase 1 — freeze backbone, train head only
get_base(model).freeze_backbone()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print('Criterion : CrossEntropyLoss  (label_smoothing=0.1, class-weighted)')
print('Optimizer : AdamW')
print('Scheduler : CosineAnnealingWarmRestarts  T0=10')
print(f'AMP       : {USE_AMP}')

## Cell 11 — Training Loop

In [ ]:
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'lr':[]}
best_val_acc      = 0.0
patience_count    = 0
backbone_unfrozen = False


def run_epoch(loader, is_train: bool):
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


header = f'{"Epoch":>6}  {"Tr Loss":>8}  {"Tr Acc":>8}  {"Vl Loss":>8}  {"Vl Acc":>8}  {"LR":>10}'
print(header)
print('─' * len(header))

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Phase 2: unfreeze backbone with differential LRs ──────────────────────
    if epoch == UNFREEZE_EPOCH and not backbone_unfrozen:
        base = get_base(model)
        base.unfreeze_backbone()
        backbone_unfrozen = True
        optimizer = optim.AdamW([
            {'params': base.backbone.parameters(),   'lr': LR_BACKBONE},
            {'params': base.classifier.parameters(), 'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-7
        )
        print(f'  ↪ Differential LR  backbone={LR_BACKBONE}  head={LR_HEAD}')

    tr_loss, tr_acc = run_epoch(train_loader, is_train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   is_train=False)
    scheduler.step(epoch)

    lr_now = optimizer.param_groups[0]['lr']
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['lr'].append(lr_now)

    tag = ''
    if vl_acc > best_val_acc:
        best_val_acc   = vl_acc
        patience_count = 0
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'val_acc':     vl_acc,
            'class_names': CLASS_NAMES,
        }, CKPT_DIR / 'best_model.pth')
        tag = '  ← best ✓'
    else:
        patience_count += 1

    print(f'{epoch:>6}  {tr_loss:>8.4f}  {tr_acc*100:>7.2f}%  '
          f'{vl_loss:>8.4f}  {vl_acc*100:>7.2f}%  {lr_now:>10.2e}{tag}')

    if patience_count >= PATIENCE:
        print(f'\n⏹  Early stopping at epoch {epoch}')
        break

print(f'\n🏆 Best val accuracy : {best_val_acc*100:.2f}%')
print(f'   Saved             → {CKPT_DIR}/best_model.pth')

## Cell 12 — Training Curves

In [ ]:
E = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Training History', fontsize=13, fontweight='bold')

axes[0].plot(E, history['train_loss'], 'b-o', ms=3, label='Train')
axes[0].plot(E, history['val_loss'],   'r-o', ms=3, label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(E, [a*100 for a in history['train_acc']], 'b-o', ms=3, label='Train')
axes[1].plot(E, [a*100 for a in history['val_acc']],   'r-o', ms=3, label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].semilogy(E, history['lr'], 'g-o', ms=3)
axes[2].set_title('Learning Rate'); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved → training_curves.png')

## Cell 13 — Load Best Model & Test Inference

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best model  epoch={ckpt['epoch']}  val_acc={ckpt['val_acc']*100:.2f}%")

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(f'\nTest Accuracy  : {accuracy_score(all_labels, all_preds)*100:.2f}%')
print(f'Test F1 (wt)   : {f1_score(all_labels, all_preds, average="weighted"):.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Cell 14 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix', fontsize=13, fontweight='bold')

for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ['d', '.2%'],
    ['Absolute Counts', 'Normalised (row)']
):
    sns.heatmap(data, ax=ax, annot=True, fmt=fmt, cmap='RdYlGn',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, linecolor='gray', annot_kws={'size': 10})
    ax.set_title(title); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved → confusion_matrix.png')

## Cell 15 — ROC Curves

In [ ]:
y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_title('ROC — One-vs-Rest', fontsize=12, fontweight='bold')

for i, (cls, col) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    auc = roc_auc_score(y_bin[:, i], all_probs[:, i])
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls}  AUC={auc:.3f}')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.text(0.55, 0.10, f'Macro AUC = {macro_auc:.4f}', fontsize=11,
        bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved → roc_curves.png')

## Cell 16 — Grad-CAM Heatmaps

In [ ]:
base = get_base(model)
# Get last block of EfficientNetV2 backbone
try:
    target_layers = [base.backbone.blocks[-1]]
except Exception:
    # Fallback: last conv layer
    convs = [m for m in base.backbone.modules() if isinstance(m, nn.Conv2d)]
    target_layers = [convs[-1]]

cam = GradCAM(model=base, target_layers=target_layers)

# Collect 3 samples per class from test set
N = 3
collected = {i: [] for i in range(NUM_CLASSES)}
base.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        probs_b = torch.softmax(base(imgs.to(DEVICE)), dim=1).cpu()
        preds_b = probs_b.argmax(1)
        for img, lbl, pred, prob in zip(imgs, labels, preds_b, probs_b):
            l = lbl.item()
            if len(collected[l]) < N:
                collected[l].append((img, l, pred.item(), prob))
        if all(len(v) >= N for v in collected.values()):
            break

fig, axes = plt.subplots(NUM_CLASSES, N * 2, figsize=(N * 5, NUM_CLASSES * 3.5))
fig.suptitle('Grad-CAM  —  CNN Attention Heatmaps', fontsize=13, fontweight='bold')

for row in range(NUM_CLASSES):
    for col_pair, (img_t, true_lbl, pred_lbl, prob) in enumerate(collected[row]):
        raw = denorm(img_t).permute(1,2,0).numpy()
        mask = cam(
            input_tensor=img_t.unsqueeze(0).to(DEVICE),
            targets=[ClassifierOutputTarget(pred_lbl)]
        )[0]
        cam_img = show_cam_on_image(raw, mask, use_rgb=True)

        c_orig = col_pair * 2
        c_cam  = col_pair * 2 + 1
        color  = 'green' if true_lbl == pred_lbl else 'red'

        axes[row][c_orig].imshow(raw)
        axes[row][c_orig].set_title(
            f'T:{CLASS_NAMES[true_lbl]}\nP:{CLASS_NAMES[pred_lbl]} {prob[pred_lbl]:.0%}',
            fontsize=7, color=color)
        axes[row][c_orig].axis('off')

        axes[row][c_cam].imshow(cam_img)
        axes[row][c_cam].set_title('Grad-CAM', fontsize=7)
        axes[row][c_cam].axis('off')

plt.tight_layout()
plt.savefig('gradcam.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved → gradcam.png')

## Cell 17 — Confidence Distribution per Class

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(14, 4), sharey=True)
fig.suptitle('Confidence Distribution per True Class', fontsize=12, fontweight='bold')

for i, (cls, ax, col) in enumerate(zip(CLASS_NAMES, axes, CLASS_COLORS)):
    mask = all_labels == i
    conf = all_probs[mask, i]
    ax.hist(conf, bins=25, color=col, edgecolor='white', alpha=0.85)
    ax.axvline(conf.mean(), color='black', linestyle='--', lw=1.5,
               label=f'μ={conf.mean():.2f}')
    ax.set_title(cls, fontweight='bold')
    ax.set_xlabel('Predicted Probability')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
axes[0].set_ylabel('Count')
plt.tight_layout()
plt.savefig('confidence_dist.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved → confidence_dist.png')

## Cell 18 — Predict a Single Image

In [ ]:
def predict_single(image_path: str):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = val_transform(image=img)['image'].unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad(), autocast(enabled=USE_AMP):
        probs = torch.softmax(model(tensor), dim=1).squeeze().cpu().numpy()
    pred = probs.argmax()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    ax1.imshow(img)
    ax1.set_title(f'{CLASS_NAMES[pred]}\n{probs[pred]:.1%} confidence',
                  fontsize=12, color=CLASS_COLORS[pred], fontweight='bold')
    ax1.axis('off')

    bars = ax2.barh(CLASS_NAMES, probs * 100, color=CLASS_COLORS, edgecolor='gray')
    for bar, p in zip(bars, probs):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{p:.1%}', va='center', fontsize=10)
    ax2.set_xlim(0, 108); ax2.set_xlabel('Probability (%)')
    ax2.set_title('Class Probabilities'); ax2.grid(alpha=0.3, axis='x')

    plt.tight_layout(); plt.show()
    return dict(zip(CLASS_NAMES, probs.tolist()))


# ── Run on first test image ───────────────────────────────────────────────────
sample_path = test_ds.samples[0][0]
print(f'Running on: {sample_path}')
result = predict_single(sample_path)
print('Probabilities:', {k: f"{v:.3f}" for k, v in result.items()})

## Cell 19 — Export (TorchScript + ONNX)

In [ ]:
base = get_base(model)
base.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# ── TorchScript trace ─────────────────────────────────────────────────────────
try:
    traced = torch.jit.trace(base, dummy)
    traced.save(str(CKPT_DIR / 'model_traced.pt'))
    print(f'✅ TorchScript → {CKPT_DIR}/model_traced.pt')
except Exception as e:
    print(f'TorchScript skipped: {e}')

# ── ONNX ──────────────────────────────────────────────────────────────────────
try:
    onnx_path = str(CKPT_DIR / 'model.onnx')
    torch.onnx.export(
        base, dummy, onnx_path,
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image':{0:'batch'}, 'logits':{0:'batch'}},
        opset_version=17,
    )
    print(f'✅ ONNX         → {onnx_path}')
except Exception as e:
    print(f'ONNX skipped: {e}  (pip install onnx)')

print('\n📁 Checkpoints:')
for f in sorted(CKPT_DIR.glob('*')):
    print(f'  {f.name:<30} {f.stat().st_size/1e6:.1f} MB')

In [ ]:
# Export test-set probabilities for the base-R evaluation script (rscript.R).
import os
import pandas as pd

CLASSES = ["No_Accident", "Minor_Accident", "Severe_Accident"]
if len(test_ds.samples) != len(all_labels) or len(all_probs) != len(all_labels):
    raise RuntimeError("Test samples and model outputs are not aligned")

os.makedirs("r_analysis", exist_ok=True)
rows = []
for (path, label), pred, probs in zip(test_ds.samples, all_preds, all_probs):
    rows.append({
        "file": os.path.basename(path),
        "true": CLASSES[label],
        "pred": CLASSES[int(pred)],
        **{f"prob_{name}": float(prob) for name, prob in zip(CLASSES, probs)},
    })

pd.DataFrame(rows).to_csv("r_analysis/predictions.csv", index=False)
print(f"Wrote {len(rows)} predictions to r_analysis/predictions.csv")
print("Run `Rscript rscript.R` from the project folder for class metrics and plots.")